# snf-peirce — Quickstart Notebook

**What this does:** Takes any CSV file, maps it to meaning, and lets you query it in plain language.

**What you need to know:** Almost nothing. Just follow the cells in order.

**How to run it:** Click a cell, press **Shift+Enter** to run it. Or click the ⏩ button at the top to run everything at once.

---

### The six dimensions

SNF organises any data into six questions:

| Dimension | Means | Examples |
|-----------|-------|----------|
| **WHO** | People, artists, organisations | attorney, artist, author, publisher |
| **WHAT** | Things, types, topics, categories | title, subject, matter type, card type |
| **WHEN** | Dates, years, time periods | year, release date, filed date |
| **WHERE** | Places, locations, labels | office, label, city, collection |
| **WHY** | Reasons, purposes, causes | matter type, reason code, genre |
| **HOW** | Methods, formats, quantities | format, condition, CMC, page count |

Once your data is mapped, you query it like this:

```
WHO.artist = "Miles Davis"
WHO.artist = "Miles Davis" AND WHEN.released BETWEEN "1955" AND "1965"
WHERE.label = "Blue Note" OR WHERE.label = "Impulse"
```

---

## Step 1 — Install and import

Run this cell once. If you've already installed snf-peirce, it'll just confirm it's there.

In [1]:
!pip install snf-peirce -q

from snf_peirce import suggest, compile_data, query, discover
import pandas as pd

print("✓ Ready.")


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✓ Ready.


---

## Step 2 — Load your data

**Option A — Use the built-in example** (recommended for your first run)

Just run the cell below as-is. It loads a small jazz record collection so you can see how everything works before trying it on your own data.

**Option B — Use your own CSV**

Replace the cell below with:
```python
df = pd.read_csv("your_file.csv")
```
Put your CSV file in the same folder as this notebook, or use the full path to it.

In [2]:
# ── OPTION A: Built-in example ────────────────────────────────────────────────
# Run this as-is for your first time through

df = pd.DataFrame({
    "artist":     ["Miles Davis",    "John Coltrane",  "Miles Davis",
                   "Bill Evans",     "Miles Davis",    "John Coltrane",
                   "Thelonious Monk","Bill Evans",     "Charles Mingus"],
    "title":      ["Kind of Blue",   "A Love Supreme", "Sketches of Spain",
                   "Waltz for Debby","Milestones",     "Blue Train",
                   "Brilliant Corners","Portrait in Jazz","Mingus Ah Um"],
    "label":      ["Columbia",       "Impulse",        "Columbia",
                   "Riverside",      "Columbia",       "Blue Note",
                   "Riverside",      "Riverside",      "Columbia"],
    "released":   ["1959", "1964", "1960", "1961", "1958",
                   "1957", "1957", "1959", "1959"],
    "format":     ["LP"] * 9,
    "release_id": ["001", "002", "003", "004", "005",
                   "006", "007", "008", "009"],
})

# ── OPTION B: Your own CSV ────────────────────────────────────────────────────
# Uncomment the line below and replace with your file path
# df = pd.read_csv("your_file.csv")

# ─────────────────────────────────────────────────────────────────────────────

print(f"✓ Loaded {len(df):,} rows, {len(df.columns)} columns")
print(f"  Columns: {list(df.columns)}")
df.head()

✓ Loaded 9 rows, 6 columns
  Columns: ['artist', 'title', 'label', 'released', 'format', 'release_id']


,artist,title,label,released,format,release_id
0,Miles Davis,Kind of Blue,Columbia,1959,LP,001
1,John Coltrane,A Love Supreme,Impulse,1964,LP,002
2,Miles Davis,Sketches of Spain,Columbia,1960,LP,003
3,Bill Evans,Waltz for Debby,Riverside,1961,LP,004
4,Miles Davis,Milestones,Columbia,1958,LP,005


---

## Step 3 — See what SNF suggests

Run this cell. SNF will look at your column names and values and suggest which dimension each column belongs to.

**You don't have to accept every suggestion.** Step 4 lets you adjust anything.

In [15]:
draft = suggest(df)
draft  # renders as a mapping table — review the suggestions

Column,Dimension,Semantic Key,Confidence,Reason
artist,WHO,artist,high,WHO-like column name
title,WHAT,title,high,WHAT-like column name
label,WHO,publisher,high,WHO-like column name
released,WHEN,released,high,date-like column name
format,HOW,format,high,HOW-like column name
release_id,WHAT,release_id,medium,high-cardinality ID-like field — nucleus candidate


---

## Step 4 — Adjust the mapping

Look at the table above. For each column, ask: **does that dimension make sense?**

If something looks wrong, use `draft.map("column_name", "dimension", "field_name")` to fix it.

**For the built-in example** — the mapping below is correct. Just run it.

**For your own data** — edit the `draft.map()` lines to match your column names.

---

### What is the nucleus?

Every record needs a **unique identifier** — the field that uniquely identifies each row.

- Record collection: release ID, catalog number
- Library catalog: ISBN, record number
- Legal matters: matter number, case ID
- Magic cards: card ID, collector number

If your data doesn't have one, you can create one: `df["id"] = range(len(df))`

In [4]:
# ── FOR THE BUILT-IN EXAMPLE: run as-is ──────────────────────────────────────

draft.map("artist",   "who",   "artist")
draft.map("title",    "what",  "title")
draft.map("label",    "where", "label")
draft.map("released", "when",  "released")
draft.map("format",   "how",   "format")

# The unique identifier for each record
draft.nucleus("release_id", prefix="music:album")

draft  # review the updated mapping

Column,Dimension,Semantic Key,Confidence,Reason
artist,WHO,artist,manual,manually mapped
title,WHAT,title,manual,manually mapped
label,WHERE,label,manual,manually mapped
released,WHEN,released,manual,manually mapped
format,HOW,format,manual,manually mapped
release_id ⬤ nucleus,WHAT,release_id,medium,high-cardinality ID-like field — nucleus candidate


---

## Step 5 — Compile

One line. This turns your mapped data into a queryable substrate.

In [5]:
lens     = draft.to_lens(lens_id="my_data", authority="me")
compiled = compile_data(df, lens)

print("✓ Compiled. Your data is ready to query.")
compiled

✓ Compiled. Your data is ready to query.


Dimension,Facts
how,9
what,18
when,9
where,9
who,9


---

## Step 6 — Discover what's in your data

Before writing queries, explore what's actually there.

In [6]:
# What dimensions does your data have?
discover(compiled, "*")

Dimension,Entities,Facts
what,9,18
who,9,9
where,9,9
when,9,9
how,9,9


In [7]:
# What values exist for WHO.artist?
discover(compiled, "WHO|artist|*")

WHO.artist,Entities
Miles Davis,3
John Coltrane,2
Bill Evans,2
Charles Mingus,1
Thelonious Monk,1


---

## Step 7 — Query

Now ask questions.

```
DIMENSION.field = "value"                          equality
DIMENSION.field BETWEEN "value1" AND "value2"      range (inclusive)
DIMENSION.field CONTAINS "text"                    partial match
query1 AND query2                                  narrows results
query1 OR query2                                   widens results
```

In [8]:
query(compiled, 'WHO.artist = "Miles Davis"')

entity_id,artist,format,label,release_id,released,title
music:album:001,Miles Davis,LP,Columbia,001,1959,Kind of Blue
music:album:003,Miles Davis,LP,Columbia,003,1960,Sketches of Spain
music:album:005,Miles Davis,LP,Columbia,005,1958,Milestones


In [9]:
# AND — narrow by adding more constraints
query(compiled, 'WHO.artist = "Miles Davis" AND WHERE.label = "Columbia"')

entity_id,artist,format,label,release_id,released,title
music:album:001,Miles Davis,LP,Columbia,001,1959,Kind of Blue
music:album:003,Miles Davis,LP,Columbia,003,1960,Sketches of Spain
music:album:005,Miles Davis,LP,Columbia,005,1958,Milestones


In [10]:
# BETWEEN — range queries
query(compiled, 'WHEN.released BETWEEN "1957" AND "1959"')

entity_id,artist,format,label,release_id,released,title
music:album:001,Miles Davis,LP,Columbia,001,1959,Kind of Blue
music:album:005,Miles Davis,LP,Columbia,005,1958,Milestones
music:album:006,John Coltrane,LP,Blue Note,006,1957,Blue Train
music:album:007,Thelonious Monk,LP,Riverside,007,1957,Brilliant Corners
music:album:008,Bill Evans,LP,Riverside,008,1959,Portrait in Jazz
music:album:009,Charles Mingus,LP,Columbia,009,1959,Mingus Ah Um


In [11]:
# OR — widen by accepting multiple values
query(compiled, 'WHERE.label = "Riverside" OR WHERE.label = "Blue Note"')

entity_id,artist,format,label,release_id,released,title
music:album:004,Bill Evans,LP,Riverside,004,1961,Waltz for Debby
music:album:006,John Coltrane,LP,Blue Note,006,1957,Blue Train
music:album:007,Thelonious Monk,LP,Riverside,007,1957,Brilliant Corners
music:album:008,Bill Evans,LP,Riverside,008,1959,Portrait in Jazz


In [12]:
# CONTAINS — partial text match
query(compiled, 'WHAT.title CONTAINS "Blue"')

entity_id,artist,format,label,release_id,released,title
music:album:001,Miles Davis,LP,Columbia,001,1959,Kind of Blue
music:album:006,John Coltrane,LP,Blue Note,006,1957,Blue Train


---

## Step 8 — Use results in pandas

Every result is a pandas DataFrame waiting to happen.

In [13]:
result = query(compiled, 'WHEN.released BETWEEN "1957" AND "1959"', limit=None)
df_result = result.to_dataframe()
df_result

,entity_id,dimension,semantic_key,value,coordinate,lens_id
0,music:album:001,how,format,LP,HOW|format|LP,my_data
1,music:album:001,what,release_id,001,WHAT|release_id|001,my_data
2,music:album:001,what,title,Kind of Blue,WHAT|title|Kind of Blue,my_data
3,music:album:001,when,released,1959,WHEN|released|1959,my_data
4,music:album:001,where,label,Columbia,WHERE|label|Columbia,my_data
5,music:album:001,who,artist,Miles Davis,WHO|artist|Miles Davis,my_data
6,music:album:005,how,format,LP,HOW|format|LP,my_data
7,music:album:005,what,release_id,005,WHAT|release_id|005,my_data
8,music:album:005,what,title,Milestones,WHAT|title|Milestones,my_data
9,music:album:005,when,released,1958,WHEN|released|1958,my_data


---

## Step 9 — Save your substrate

Save your compiled substrate so you don't have to recompile every time.

In [14]:
compile_data(df, lens, into="csv://my_spoke")
print("✓ Saved to my_spoke/")
print("  Reload later with: python shell.py csv://my_spoke")

✓ Saved to my_spoke/
  Reload later with: python shell.py csv://my_spoke


---

## ✓ You're done

You just took a CSV, mapped it to meaning once, and queried it in plain language. No SQL. No schema knowledge. No joins.

---

### Works for any domain

| Domain | WHO | WHAT | WHEN | WHERE |
|--------|-----|------|------|-------|
| Record collection | artist | title, format | year | label |
| Library catalog | author | subject, title | pub year | publisher |
| Legal matters | attorney, client | matter type | year | office |
| Magic cards | artist | card type, color | set year | set |
| Museum collection | artist, maker | medium, subject | period | location |

The query syntax is identical regardless of domain. Only the field names change.

---

**More:**
- [snf-peirce on PyPI](https://pypi.org/project/snf-peirce/)
- [GitHub](https://github.com/peirce-lang/snf-peirce)